In [32]:
!pip install torchvision av

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 39.8 MB/s eta 0:00:00


In [102]:
import pandas as pd
import transformers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import numpy as np
import torch
from transformers import  Trainer, TrainingArguments , VideoMAEForVideoClassification , VideoMAEImageProcessor
from torch.utils.data import Dataset


In [103]:
id2label = {
    0: "no_event",
    1: "ball_hit",
    2: "ball_bounced",
}

label2id = {v: k for k, v in id2label.items()}

In [104]:
from huggingface_hub import login
login()

In [105]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [106]:
import os
folders = ['Layal_vs_Fery' , 'Layal_vs_Hsu' , 'Layal_vs_Martin' , 'daderi_vs_coria' , 'novak_vs_thiem'
           , 'stan_vs_rafa' , 'rafa_vs_medvedev' , 'paolini_vs_pegula']
files_list = []

for folder in folders :
    paths = os.listdir(os.path.join('/content/drive/MyDrive' ,'tennis events detection','videos_dataset', folder ))
    full_paths = [os.path.join('/content/drive/MyDrive' ,'tennis events detection', 'videos_dataset' , folder , path) for path in paths ]
    files_list.extend(full_paths)

In [107]:
train_df = pd.DataFrame(files_list, columns=['file_path'])
train_df['label'] = train_df['file_path'].apply(lambda x: "_".join(x.split('.')[0].split("_")[-2:]))


In [108]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2292 entries, 0 to 2291
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   file_path  2292 non-null   object
 1   label      2292 non-null   object
dtypes: object(2)
memory usage: 35.9+ KB


In [109]:
train_df = train_df[train_df['label'].isin(id2label.values())]

In [110]:
train_df['label'] = train_df['label'].map(label2id)

In [111]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2288 entries, 0 to 2291
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   file_path  2288 non-null   object
 1   label      2288 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 53.6+ KB


In [112]:
from sklearn.model_selection import train_test_split

train_paths, val_paths = train_test_split(
    train_df['file_path'].tolist(),
    test_size=0.2,
    shuffle=True,
    random_state=42 ,
    stratify=train_df['label'].tolist()
)

In [113]:
filename = train_paths[20].split("/")[-1].split(".")[0]
label = "_".join(filename.split("_")[-2:])

In [114]:
label

'ball_hit'

In [115]:
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset

class VideoDataset(Dataset):

    def __init__(self, file_paths, num_frames=16):
        self.file_paths = file_paths
        self.num_frames = num_frames

    def _read_video(self, path):
        cap = cv2.VideoCapture(path)
        frames = []
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            # OpenCV reads BGR -> convert to RGB
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)
        cap.release()
        return frames

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        path = self.file_paths[idx]
        filename = path.split("/")[-1].split(".")[0]
        label = "_".join(filename.split("_")[-2:])

        # Handle potential KeyError if label not in label2id
        if label not in label2id:
            print(f"Warning: Label '{label}' not found in label2id for video: {path}. Skipping this sample.")
            return None # Return None for invalid labels

        label_id = label2id[label]
        frames = self._read_video(path)

        if not frames: # If video reading failed or video was empty
            print(f"Warning: Could not read frames for video: {path}. Skipping this sample.")
            return None # Return None for unreadable videos

        n = len(frames)
        if n > self.num_frames:
            # Uniform sampling
            indices = np.linspace(
                0,
                n - 1,
                self.num_frames,
                dtype=int,
            )
            frames = [frames[i] for i in indices]
        elif n < self.num_frames:
            # Duplicate last frame
            while len(frames) < self.num_frames:
                frames.append(frames[-1].copy())

        return {
            "video": frames,      # list of RGB numpy arrays
            "label": label_id,
        }

In [116]:
from dataclasses import dataclass
import torch

@dataclass
class VideoDataCollator:

    processor: VideoMAEImageProcessor

    def __call__(self, batch):
        # Filter out None values which indicate unreadable/invalid samples from VideoDataset
        batch = [item for item in batch if item is not None]

        if not batch: # If all samples were invalid in this batch
            return None # Return None or an empty dictionary if batch is empty after filtering

        videos = [x["video"] for x in batch]

        labels = torch.tensor(
            [x["label"] for x in batch],
            dtype=torch.long,
        )

        encoding = self.processor(
            videos,
            return_tensors="pt",
        )

        encoding["labels"] = labels

        return encoding

In [117]:
train_dataset = VideoDataset(train_paths)
validation_dataset = VideoDataset(val_paths )

In [118]:
from transformers import VideoMAEImageProcessor
from transformers import VideoMAEForVideoClassification

processor = VideoMAEImageProcessor.from_pretrained(
    "MCG-NJU/videomae-base-finetuned-kinetics"
)

model = VideoMAEForVideoClassification.from_pretrained(
    "MCG-NJU/videomae-base-finetuned-kinetics",
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `400`.


Loading weights:   0%|          | 0/162 [00:00<?, ?it/s]

[transformers] VideoMAEForVideoClassification LOAD REPORT from: MCG-NJU/videomae-base-finetuned-kinetics
Key                                                            | Status     |                                                                                         
---------------------------------------------------------------+------------+-----------------------------------------------------------------------------------------
videomae.encoder.layer.{0...11}.attention.attention.v_bias     | UNEXPECTED |                                                                                         
videomae.encoder.layer.{0...11}.attention.attention.q_bias     | UNEXPECTED |                                                                                         
videomae.encoder.layer.{0...11}.attention.attention.value.bias | MISSING    |                                                                                         
videomae.encoder.layer.{0...11}.attention.attention.key.bias

In [119]:
collator = VideoDataCollator(processor)

In [120]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import torch

labels = train_df["label"].to_numpy()

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(labels),
    y=labels,
)

class_weights = torch.tensor(weights, dtype=torch.float)
print(class_weights)

tensor([0.4476, 2.7141, 2.5171])


In [126]:
training_args = TrainingArguments(

    output_dir="checkpoints",

    learning_rate=2e-5,

    per_device_train_batch_size=4,

    per_device_eval_batch_size=4,

    num_train_epochs=20,

    weight_decay=0.01,

    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="f1",

    fp16=True,
    remove_unused_columns=False,
)

In [127]:
import torch.nn as nn
from transformers import Trainer

class WeightedTrainer(Trainer):

    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):

        labels = inputs.pop("labels")

        outputs = model(**inputs)

        logits = outputs.logits

        loss_fn = nn.CrossEntropyLoss(
            weight=self.class_weights.to(logits.device)
        )

        loss = loss_fn(logits, labels)

        if return_outputs:
            return loss, outputs

        return loss

In [128]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_recall_fscore_support

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    preds = logits.argmax(axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(

        labels,
        preds,
        average="macro",
        zero_division=0,
    )

    accuracy = accuracy_score(labels, preds)

    return {

        "accuracy": accuracy,

        "precision": precision,

        "recall": recall,

        "f1": f1,
    }

In [129]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    data_collator=collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss
